In [593]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from DateTimeEncoder.DateEncoder import DateEncoder
from DateTimeEncoder.DurationEncoder import DurationEncoder
from DateTimeEncoder.TimeEncoder import TimeEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np


from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [594]:
df = pd.read_csv("flights.csv")

In [595]:
print(df.columns)

Index(['index', 'Airline', 'Date_of_Journey', 'Source', 'Destination',
       'Dep_Time', 'Duration', 'Total_Stops', 'Additional_Info', 'Price'],
      dtype='str')


In [596]:
# Combine Source and Destination into a single Route feature
df["Route"] = df["Source"].astype(str) + "_" + df["Destination"].astype(str)

In [597]:
categorical_cols = ["Airline", "Additional_Info"]
destination_cols = ["Source", "Destination"]
duration_cols = ["Duration"]
time_cols = ["Dep_Time"]
stops_order = [["non-stop", "1 stop", "2 stops", "3 stops", "4 stops"]]
date_cols = ["Date_of_Journey"]

In [598]:
# Duplicate Categories in Additional_Info
df["Additional_Info"] = df["Additional_Info"].replace("No info", "No Info")
# Count occurrences of each category
info_counts = df["Additional_Info"].value_counts()

# Identify categories that appear infrequently (e.g., fewer than 10 times)
rare_categories = info_counts[info_counts < 10].index

# 4. Replace rare categories with 'Other'
df["Additional_Info"] = df["Additional_Info"].replace(rare_categories, "Other")

In [599]:
X = df.drop(columns=["Price","index"])
y = df["Price"]
X.columns

Index(['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Dep_Time',
       'Duration', 'Total_Stops', 'Additional_Info', 'Route'],
      dtype='str')

In [600]:
def check_is_weekend(X):
    # Converts dayofweek (0-6) to 1 if Friday (4), Saturday (5), or Sunday (6), else 0
    return (X >= 4).astype(int)

def weekend_feature_names(transformer, input_features):
    return ["Date_of_Journey_is_weekend"]


# Pipeline step for indicating weekends
is_weekend_transformer = Pipeline(
    steps=[
        ("date_enc", DateEncoder(features=["dayofweek"])),
        (
            "to_weekend",
            FunctionTransformer(
                check_is_weekend, feature_names_out=weekend_feature_names
            ),
        ),
    ])


In [601]:
#Prepocess tranform
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                feature_name_combiner=lambda feature, category: f"{category}",
            ),
            categorical_cols,
        ),
        (
            "destination",
            OneHotEncoder(
                handle_unknown="ignore",
                feature_name_combiner=lambda feature, category: f"{feature}_{category}",
            ),
            destination_cols,
        ),
        (
            "date",
            DateEncoder(features=["month", "day", "dayofweek"]),
            date_cols,
        ),
        ("weekend", is_weekend_transformer, date_cols),
        ("time", TimeEncoder(features=["hour", "minute"]), time_cols),
        ("duration", DurationEncoder(), duration_cols),
        (
            "stops",
            OrdinalEncoder(
                categories=stops_order,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ["Total_Stops"],
        ),
    ],
    remainder="drop",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

In [602]:
processed_matrixs=preprocessor.fit_transform(X)

In [603]:
cols = preprocessor.get_feature_names_out()
num_columns = 3

for i in range(0, len(cols), num_columns):
    row = cols[i : i + num_columns]
    print("".join(f"{item:<35}" for item in row))

Air Asia                           Air India                          GoAir                              
IndiGo                             Jet Airways                        Jet Airways Business               
Multiple carriers                  Multiple carriers Premium economy  SpiceJet                           
Trujet                             Vistara                            Vistara Premium economy            
1 Long layover                     In-flight meal not included        No Info                            
No check-in baggage included       Other                              Source_Banglore                    
Source_Chennai                     Source_Delhi                       Source_Kolkata                     
Source_Mumbai                      Destination_Banglore               Destination_Cochin                 
Destination_Delhi                  Destination_Hyderabad              Destination_Kolkata                
Destination_New Delhi              Date_of_Jou

In [604]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [605]:
#Defining the model
model = RandomForestRegressor()

In [606]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [607]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['Airline','Date_of_Journey','Source',...,'Total_Stops','Additional_Info', 'Route']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('destination', ...), ...]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0
,"verbose_feature_names_out verbose

In [608]:
pre_vals = pipeline.predict(X_valid)

In [610]:
mean_absolute_error(y_valid, pre_vals)

666.0149694108339